# 01 — LV Scar Exploration

**Focus:** Core Zone (CZ) only.  
The CZ is the dense fibrotic core; its negative mask approximates the Border Zone and conduction channels.

**Pipeline**
1. Scan dataset → keep only cases that have a `Core Surface.vtk`
2. 80 / 20 split → *Known* set (working set) and *Held-out* set (untouched)
3. **3-D visualisation** of the Known set — overlayed population view + per-patient grid
4. **2-D cylindrical polar projection** — individual bull's-eye maps + superposition
5. **ISOMAP shape analysis** — per-scar 2-D shape embedding

## 0 · Imports & config

In [ ]:
import json, random, sys
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv
from sklearn.manifold import Isomap
from tqdm import tqdm

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
from data_loading import scan, PatientCase

# ── config ────────────────────────────────────────────────────────────────────
HD_ROOT        = r"F:/RM_TEKNON_DEVELOP"
SEED           = 42
TRAIN_RATIO    = 0.80
N_R            = 64     # radial bins in polar map   (apex → base)
N_THETA        = 128    # angular bins               (0° → 360°)
GRID3D_MAX     = 24     # max patients shown in the 3-D per-patient grid
ISOMAP_PTS     = 800    # max vertices fed to ISOMAP per mesh (subsampled if larger)
ISOMAP_K       = 10     # ISOMAP n_neighbors

RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

pv.set_jupyter_backend("static")
plt.rcParams["figure.dpi"] = 120

## 1 · Scan & filter

In [ ]:
all_cases = scan(HD_ROOT)
valid = [
    c for c in all_cases
    if c.tissue_surfaces.get("core") is not None
    and c.tissue_surfaces["core"].exists()
]
print(f"Total scanned: {len(all_cases)}   |   with CZ: {len(valid)}")
for yr, n in sorted(Counter(c.year for c in valid).items()):
    print(f"  {yr}: {n}")

## 2 · 80 / 20 split

In [ ]:
random.seed(SEED)
shuffled  = random.sample(valid, len(valid))
n_known   = int(len(shuffled) * TRAIN_RATIO)
known     = shuffled[:n_known]
held_out  = shuffled[n_known:]

(RESULTS / "split.json").write_text(json.dumps({
    "seed": SEED, "train_ratio": TRAIN_RATIO,
    "known":    [f"{c.year}/{c.patient_id}" for c in known],
    "held_out": [f"{c.year}/{c.patient_id}" for c in held_out],
}, indent=2))
print(f"Known: {len(known)}   Held-out: {len(held_out)}   → results/split.json")

---
## 3 · 3-D visualisation

Each render shows **two meshes stacked**:
- **Myocardium** (or best available LV shell) — semi-transparent gray shell → anatomical context
- **Core Zone** — opaque red → the scar, visible *inside* the shell

We produce two views:
- **3a** — all Known patients overlayed in one scene (population footprint)
- **3b** — one thumbnail per patient (individual inspection)

In [ ]:
def _best_shell(case: PatientCase):
    """Load the most complete LV shell available, or None."""
    for key in ("myocardium", "lv", "endo", "epi"):
        p = case.anatomy.get(key)
        if p is not None and p.exists():
            return pv.read(str(p))
    return None

### 3a — Overlayed population view

In [ ]:
cmap_pop = plt.get_cmap("tab20", len(known))
pl = pv.Plotter(off_screen=True, window_size=(900, 700))
pl.set_background("white")

ok_3d = []
for i, case in enumerate(tqdm(known, desc="3-D overlay")):
    try:
        shell = _best_shell(case)
        cz    = pv.read(str(case.tissue_surfaces["core"]))
        if shell is not None:
            pl.add_mesh(shell, color="lightgray", opacity=0.08,
                        smooth_shading=True, show_scalar_bar=False)
        pl.add_mesh(cz, color=cmap_pop(i)[:3], opacity=0.75,
                    smooth_shading=True, show_scalar_bar=False)
        ok_3d.append(case)
    except Exception as e:
        print(f"  skip {case.year}/{case.patient_id}: {e}")

pl.view_isometric()
img_overlay = pl.screenshot(None, return_img=True)
pl.close()

fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(img_overlay); ax.axis("off")
ax.set_title(f"3-D overlay — Known CZ surfaces  (n={len(ok_3d)})", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS / "3d_overlay_known.png", bbox_inches="tight")
plt.show()

### 3b — Per-patient 3-D grid

One thumbnail per patient: gray myocardium shell + red CZ inside.  
Capped at `GRID3D_MAX` patients to keep render time manageable.

In [ ]:
subset   = known[:GRID3D_MAX]
ncols_3d = 6
nrows_3d = int(np.ceil(len(subset) / ncols_3d))

thumbnails = []
for case in tqdm(subset, desc="3-D grid renders"):
    try:
        pl = pv.Plotter(off_screen=True, window_size=(300, 260))
        pl.set_background("#1e1e1e")
        shell = _best_shell(case)
        cz    = pv.read(str(case.tissue_surfaces["core"]))
        if shell is not None:
            pl.add_mesh(shell, color="lightgray", opacity=0.15,
                        smooth_shading=True, show_scalar_bar=False)
        pl.add_mesh(cz, color="crimson", opacity=0.95,
                    smooth_shading=True, show_scalar_bar=False)
        pl.view_isometric()
        thumbnails.append((case, pl.screenshot(None, return_img=True)))
        pl.close()
    except Exception as e:
        print(f"  skip {case.year}/{case.patient_id}: {e}")

fig, axes = plt.subplots(nrows_3d, ncols_3d,
                         figsize=(ncols_3d * 2.2, nrows_3d * 2))
fig.suptitle(f"3-D per-patient grid — first {len(thumbnails)} of Known set",
             fontsize=12)
axes_flat = axes.flatten()
for ax in axes_flat:
    ax.axis("off")
for ax, (case, img) in zip(axes_flat, thumbnails):
    ax.imshow(img)
    ax.set_title(f"{case.year}\n{case.patient_id}", fontsize=6)

plt.tight_layout()
plt.savefig(RESULTS / "3d_grid_known.png", bbox_inches="tight")
plt.show()

---
## 4 · 2-D cylindrical polar projection

### How the projection works — step by step

The LV is a thick-walled, elongated shape (like a rugby ball cut in half).  
The scar (CZ) is a 3-D surface mesh living *inside* the wall.  
We want to represent this on a flat circle so we can compare patients.

---

**Step 1 — Find the long axis of the LV (via PCA)**

We load the full LV shell (myocardium) and run PCA on its vertex coordinates.  
PCA finds the direction along which the point cloud spreads the most.  
Because the LV is elongated apex-to-base, that direction *is* the long axis.  
We also identify the **apex** as the most extreme point along that axis (the pointy tip).

```
Why not use the CZ mesh for PCA?
The scar fragment is a small, irregular patch — its own PCA would give a
meaningless axis. We need the full LV geometry to define 'up'.
```

---

**Step 2 — Cylindrical coordinates**

Once we have the axis, every point in 3-D space can be described by two numbers:

| Coordinate | How to compute | What it means |
|---|---|---|
| `s` (longitudinal) | dot-product of (point − apex) with the axis, normalised 0→1 | 0 = apex tip, 1 = base opening |
| `θ` (circumferential) | `atan2` of the point projected onto the plane perpendicular to the axis | which 'clock position' around the wall |

Concretely, for each vertex `p` of the CZ mesh:
```
v  = p − apex                   # vector from apex to this point
s  = (v · axis) / total_length  # how far along the axis
perp = v − s_raw * axis         # component perpendicular to axis
θ  = atan2(perp · w, perp · u)  # angle in the transverse plane
```
where `u` and `w` are two fixed perpendicular directions in the transverse plane.

---

**Step 3 — Rasterise onto a 2-D grid**

We bin `s` into `N_R` radial slots and `θ` into `N_THETA` angular slots.  
Any bin that contains at least one CZ vertex is marked True (scar present).

---

**Step 4 — Plot as a bull's-eye**

The grid is displayed as a polar (circular) chart:
- **Centre** = apex (s = 0)
- **Outer ring** = base (s = 1)
- **Angle** = position around the wall (anterior, lateral, inferior, septal)

---

**Known limitation — the apex singularity**

The apex is a dome — its 3-D surface area is tiny, but in the polar map it occupies
the entire inner disc. A small apical scar therefore appears artificially spread out
near the centre. This is the same distortion as Antarctica on a Mercator map.
For *location* comparison across patients this is acceptable; for *shape* analysis
of apical scars it is not → that is why we use ISOMAP in Section 5.

https://www.youtube.com/shorts/2QyCR4-fzO4

In [ ]:
# ── projection helpers ────────────────────────────────────────────────────────

def _long_axis(mesh: pv.PolyData):
    """
    PCA on mesh vertices → (unit long-axis, apex point).
    axis points from apex toward base.
    """
    pts    = np.array(mesh.points)
    center = pts.mean(axis=0)
    _, _, Vt = np.linalg.svd(pts - center, full_matrices=False)
    axis   = Vt[0]                              # direction of max variance
    proj   = (pts - center) @ axis
    apex   = pts[proj.argmin()]                 # most extreme negative end = tip
    if np.dot(center - apex, axis) < 0:         # ensure axis points apex → base
        axis = -axis
    return axis, apex


def polar_map(case: PatientCase,
              n_r: int = N_R, n_theta: int = N_THETA) -> np.ndarray:
    """
    Project the Core Zone to a 2-D polar grid.

    Returns bool (n_r, n_theta):
        row 0  = apex
        row -1 = base
    """
    # --- Step 1: load reference for axis estimation
    ref = _best_shell(case)
    if ref is None:                             # fallback: use CZ itself
        ref = pv.read(str(case.tissue_surfaces["core"]))
    axis, apex = _long_axis(ref)

    # --- Step 2: build orthonormal transverse frame (axis, u, w)
    secondary = np.array([0., 0., 1.])
    if abs(np.dot(axis, secondary)) > 0.9:
        secondary = np.array([1., 0., 0.])
    u = np.cross(axis, secondary);  u /= np.linalg.norm(u)
    w = np.cross(axis, u)

    # --- Step 3: project each CZ vertex
    cz    = pv.read(str(case.tissue_surfaces["core"]))
    pts   = np.array(cz.points)
    v     = pts - apex
    s_raw = v @ axis
    span  = s_raw.max() - s_raw.min()
    s     = np.clip((s_raw - s_raw.min()) / (span + 1e-9), 0., 1.)
    perp  = v - np.outer(s_raw, axis)
    theta = np.arctan2(perp @ w, perp @ u) % (2 * np.pi)

    # --- Step 4: rasterise
    grid = np.zeros((n_r, n_theta), dtype=bool)
    ri   = np.clip((s     * n_r   ).astype(int), 0, n_r     - 1)
    ti   = np.clip((theta / (2*np.pi) * n_theta).astype(int), 0, n_theta - 1)
    grid[ri, ti] = True
    return grid


def _draw_bullseye(grid, ax, title="", cmap="Reds", vmax=1.):
    """Render one polar grid onto a matplotlib polar Axes."""
    n_r, n_theta = grid.shape
    T, R = np.meshgrid(
        np.linspace(0, 2*np.pi, n_theta + 1),
        np.linspace(0, 1,       n_r     + 1),
    )
    ax.pcolormesh(T, R, grid.astype(float),
                  cmap=cmap, vmin=0, vmax=vmax, shading="flat")
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_rticks([]); ax.set_xticks([])
    if title:
        ax.set_title(title, fontsize=7, pad=2)

print("Helpers ready.")

### 4a — Compute all polar maps

In [ ]:
pmaps   = {}      # key → np.ndarray
failed  = []
for case in tqdm(known, desc="Polar projection"):
    key = f"{case.year}/{case.patient_id}"
    try:
        pmaps[key] = polar_map(case)
    except Exception as e:
        failed.append(key)
        print(f"  skip {key}: {e}")
print(f"OK: {len(pmaps)}   failed: {len(failed)}")

### 4b — Individual polar maps (per-patient grid)

In [ ]:
keys  = list(pmaps.keys())
ncols = 8
nrows = int(np.ceil(len(keys) / ncols))

fig = plt.figure(figsize=(ncols * 1.6, nrows * 1.6))
fig.suptitle(f"Individual CZ polar maps — Known set  (n={len(keys)})",
             fontsize=12, y=1.01)
for idx, key in enumerate(keys):
    ax = fig.add_subplot(nrows, ncols, idx + 1, projection="polar")
    _draw_bullseye(pmaps[key], ax, title=key.split("/")[1])
for idx in range(len(keys), nrows * ncols):
    fig.add_subplot(nrows, ncols, idx + 1).axis("off")

plt.tight_layout()
plt.savefig(RESULTS / "polar_grid_known.png", bbox_inches="tight")
plt.show()

### 4c — Superposition: scar density across the cohort

Each cell shows the **fraction of Known patients** that have CZ scar at that location.  
Left panel: binary footprint (any patient). Right: continuous density.

In [ ]:
stack   = np.stack(list(pmaps.values()))   # (N, N_R, N_THETA)
density = stack.mean(axis=0)               # fraction in [0, 1]

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(11, 5),
                                  subplot_kw={"projection": "polar"})
fig.suptitle(f"CZ superposition — Known set  (n={len(pmaps)})", fontsize=13)

_draw_bullseye(density > 0, ax_l, title="Any-patient scar footprint")
_draw_bullseye(density,     ax_r, title="Scar density (fraction of patients)",
               cmap="hot_r", vmax=density.max())

sm   = plt.cm.ScalarMappable(cmap="hot_r",
                              norm=plt.Normalize(0, density.max()))
sm.set_array([])
fig.colorbar(sm, ax=ax_r, pad=0.12, fraction=0.046,
             label="Fraction of patients")
plt.tight_layout()
plt.savefig(RESULTS / "polar_superposition_known.png", bbox_inches="tight")
plt.show()

---
## 5 · ISOMAP — shape analysis of individual CZ scars

### Why ISOMAP here and not for location?

The cylindrical projection tells you *where* the scar is on the LV wall.  
ISOMAP tells you *what shape* the scar has, regardless of where it lives.

ISOMAP measures **geodesic distances** between points — shortest paths
*crawling along the mesh surface*, not straight lines through 3-D space.  
This makes it curvature-aware: a small apical scar (on a highly curved dome)
stays small; a mid-wall scar (on a flatter surface) also stays its true size.
The cylindrical projection cannot do this — it distorts apex geometry.

The output is a 2-D scatter of CZ vertices where **distances reflect the
intrinsic surface geometry of each scar** — its elongation, fragmentation,
and connectivity are all visible.

**Important caveat:** each patient's ISOMAP embedding is in its own arbitrary
coordinate frame (rotations and reflections are not fixed). ISOMAP embeddings
cannot be directly overlaid across patients without a registration step.  
In this notebook we use them for *visual inspection* of individual scar shapes.
Cross-patient shape comparison will be developed in a later notebook.

In [ ]:
def isomap_embedding(case: PatientCase,
                     n_pts: int = ISOMAP_PTS,
                     k: int     = ISOMAP_K) -> np.ndarray:
    """
    Run ISOMAP on the CZ mesh vertices.

    Large meshes are randomly subsampled to n_pts vertices so the
    O(n²) distance matrix stays tractable.

    Returns
    -------
    np.ndarray, shape (n_pts_actual, 2)
    """
    cz   = pv.read(str(case.tissue_surfaces["core"]))
    pts  = np.array(cz.points)
    if len(pts) > n_pts:
        idx = np.random.default_rng(SEED).choice(len(pts), n_pts, replace=False)
        pts = pts[idx]
    emb = Isomap(n_neighbors=k, n_components=2).fit_transform(pts)
    return emb

### 5a — Individual ISOMAP embeddings (grid)

In [ ]:
embeddings = {}
for case in tqdm(known, desc="ISOMAP"):
    key = f"{case.year}/{case.patient_id}"
    try:
        embeddings[key] = isomap_embedding(case)
    except Exception as e:
        print(f"  skip {key}: {e}")
print(f"OK: {len(embeddings)}")

In [ ]:
ikeys = list(embeddings.keys())
ncols = 8
nrows = int(np.ceil(len(ikeys) / ncols))

fig, axes = plt.subplots(nrows, ncols,
                         figsize=(ncols * 1.8, nrows * 1.8))
fig.suptitle(f"ISOMAP shape embeddings — Known CZ  (n={len(ikeys)})",
             fontsize=12, y=1.01)
axes_flat = axes.flatten()
for ax in axes_flat:
    ax.axis("off")
for ax, key in zip(axes_flat, ikeys):
    emb = embeddings[key]
    ax.scatter(emb[:, 0], emb[:, 1],
               s=1, c="crimson", alpha=0.5, linewidths=0)
    ax.set_aspect("equal")
    ax.set_title(key.split("/")[1], fontsize=6)
    ax.axis("on")
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)

plt.tight_layout()
plt.savefig(RESULTS / "isomap_grid_known.png", bbox_inches="tight")
plt.show()

---
## Summary of outputs

| File | Content |
|---|---|
| `results/split.json` | Frozen Known / Held-out patient lists |
| `results/3d_overlay_known.png` | All Known CZ surfaces overlayed in 3-D |
| `results/3d_grid_known.png` | Per-patient 3-D thumbnails (first `GRID3D_MAX`) |
| `results/polar_grid_known.png` | Individual 2-D bull's-eye per patient |
| `results/polar_superposition_known.png` | Population scar density map |
| `results/isomap_grid_known.png` | Per-scar ISOMAP shape embedding |